In [1]:
import json
import sys
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
CURRICULUM_FOLDER = REPO_ROOT / "raw_data" / "curricula"
COURSE_FOLDER = REPO_ROOT / "raw_data" / "training_courses"
SUPPLEMENTARY_FOLDER = REPO_ROOT / "raw_data" / "supplementary"
OUTPUT_FOLDER = REPO_ROOT / "methodology" / "output"
PRIMARY_VARIANT = "overview_learn_takeaways"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run the notebooks of data_load first"

from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
RUN = study.run_id
storage = Storage.of(study)
print("database:", DB_PATH.relative_to(REPO_ROOT))
print("run     :", RUN)

database: database\CSCAF_public.db
run     : sat-12l-sm_all-mpnet-base-v2_414c4542


In [2]:
def month(day):
    return date.fromisoformat(day).strftime("%B %Y")


editions = Counter(storage.curricula.editions())
curricula = {
    "curricula": storage.curricula.count(),
    "countries": storage.curricula.count_countries(),
    "courses": storage.curricula.count_courses(),
    "sentences": storage.curricula.count_sentences(RUN),
    "editions": dict(editions.most_common()),
    "most_common_edition": editions.most_common(1)[0][0],
}
first, last = storage.advertisements.posting_period()
advertisements = {
    "advertisements": storage.advertisements.count(),
    "companies": storage.advertisements.count_companies(),
    "countries": storage.advertisements.count_countries(),
    "first_posted": first,
    "last_posted": last,
    "period": f"{month(first)} - {month(last)}",
    "sentences": storage.advertisements.count_sentences(RUN),
    "without_sentence": storage.advertisements.count_without_sentences(RUN),
}
first, last = storage.trainings.access_period()
trainings = {
    "courses": storage.trainings.count_courses(),
    "variants": storage.trainings.variant_counts(),
    "first_accessed": first,
    "last_accessed": last,
    "period": month(first) if month(first) == month(last) else f"{month(first)} - {month(last)}",
    "sentences_primary": storage.trainings.count_sentences(RUN, PRIMARY_VARIANT),
}
table_1 = {"curricula": curricula, "advertisements": advertisements, "trainings": trainings}
table_1

{'curricula': {'curricula': 27,
  'countries': 27,
  'courses': 421,
  'sentences': 6910,
  'editions': {'2026/27': 17, '2025/26': 9, '2024/25': 1},
  'most_common_edition': '2026/27'},
 'advertisements': {'advertisements': 5136,
  'companies': 2118,
  'countries': 27,
  'first_posted': '2025-03-05',
  'last_posted': '2026-09-18',
  'period': 'March 2025 - September 2026',
  'sentences': 215747,
  'without_sentence': 0},
 'trainings': {'courses': 22,
  'variants': {'overview': 22, 'overview_learn_takeaways': 22},
  'first_accessed': '2026-10-04',
  'last_accessed': '2026-10-11',
  'period': 'October 2026',
  'sentences_primary': 789}}

In [3]:
rows = storage.curricula.programmes()
table_s1 = [{"country": country, "university": university, "programme": " ".join(programme.split()),
             "edition": edition, "courses": courses, "curriculum_key": key}
            for country, university, programme, edition, courses, key in rows]
print(f"{'country':12s} {'edition':8s} {'courses':>7s}  university | programme (as stored)")
for row in table_s1:
    print(f"{row['country']:12s} {row['edition']:8s} {row['courses']:7d}  {row['university']} | {row['programme']}")

country      edition  courses  university | programme (as stored)
Austria      2026/27       16  University of Applied Sciences St. Pölten (USTP) | Cyber Security and Resilience (MSc)
Belgium      2026/27       21  KU Leuven | Master of Cybersecurity (advanced master)
Bulgaria     2026/27       15  New Bulgarian University | Cyber Security (master, taught in Bulgarian)
Croatia      2026/27       23  Zagreb University of Applied Sciences (TVZ) | Information Security and Digital Forensics (specialist graduate professional study, Master of Engineering)
Cyprus       2026/27        8  Open University of Cyprus | Computer and Network Security (MSc)
Czechia      2026/27       12  Czech Technical University in Prague, Faculty of Information Technology | Computer Security (master specialisation, Informatics)
Denmark      2026/27       17  Aalborg University Copenhagen | Cyber Security (MSc in Engineering)
Estonia      2025/26       28  Tallinn University of Technology (TalTech) | Cybersecurity 

In [4]:
pairs = Counter((" ".join((company or "").split()), description)
                for company, description in storage.advertisements.companies_and_descriptions())
reposted = sum(n for n in pairs.values() if n > 1)
repeated = {"advertisements": reposted, "share": reposted / advertisements["advertisements"],
            "descriptions": sum(1 for n in pairs.values() if n > 1)}
repeated

{'advertisements': 1669, 'share': 0.32496105919003115, 'descriptions': 558}

In [5]:
CYBERHEAD_ENTRY = {
    "at_ustp": ("aut", "b19f270e03ef46fc8a7850fec77d46c9"), "be_kuleuven": ("bel", "a50f6cba494d4d28aee7336421379a71"),
    "cy_ouc": ("cyp", "1d4bcd7f1e8d466d813339bc83b59441"), "dk_aau": ("dnk", "5a8ff01336204efeaf05d9995f394751"),
    "fi_jamk": ("fin", "d594e73891a84ddb9fc84c5134466e5f"), "fr_ubs": ("fra", "8e5df4eca30e4e968756ad5ed7e97764"),
    "de_hdbw": ("deu", "efa71c32e5d446bb8a22fa1e733f80e4"), "gr_ihu": ("grc", "bcd4012bc3e94f1a9f04b3df5933ed64"),
    "hu_nke": ("hun", "f0566942b9bc42dda6c04361c08c6084"), "ie_mtu": ("irl", "f14965c0b12d4a669403872067cbe845"),
    "it_unipi": ("ita", "1de3f0a4136f436a9510a2eca08ae710"), "lv_lu": ("lva", "d444ce8ceae1464d81dc18dbc0910e2c"),
    "mt_um": ("mlt", "777f8d316de6453dbea44d76c1171fef"), "pl_prz": ("pol", "e663f5cafc674eec9fcb634b17d5b833"),
    "pt_ipleiria": ("prt", "b9480583d048407883473c67cfd72321"), "ro_utcn": ("rou", "86c8659261b74455a577e9dfa9dff0d9"),
    "sk_tuke": ("svk", "8d105dc71aa240c49a63b50188d5345a"), "si_fis": ("svn", "95c6840d25504be1851b32bbe5913dd4"),
    "es_uah": ("esp", "91bd9b7fab0642a7bde866d2bf457ee7"), "se_kth": ("swe", "723abd49b70b453b882fa4f96d773cbb"),
}
snapshot = json.loads((SUPPLEMENTARY_FOLDER / "cyberhead_programmes.json").read_text(encoding="utf-8"))
entries = {r["ID"]: r["fields"] for r in snapshot["data"]["programmes"]["all_programmes"]["results"]}
keys = [row["curriculum_key"] for row in table_s1]
assert set(CYBERHEAD_ENTRY) <= set(keys)
for key in keys:
    if key not in CYBERHEAD_ENTRY:
        print(f"{key:15s} not in CyberHEAD")
        continue
    country, entry_id = CYBERHEAD_ENTRY[key]
    fields = entries[entry_id]
    assert fields["country"] == country and fields["programme_type"] == "Master Degree", (key, fields["country"])
    print(f"{key:15s} {fields['country']}  {fields['institution_name'][:45]:45s} | {fields['programme_name'][:55]}")
cyberhead = {"entries_in_snapshot": len(entries), "programmes_in_cyberhead": len(CYBERHEAD_ENTRY),
             "not_in_cyberhead": sorted(set(keys) - set(CYBERHEAD_ENTRY))}
cyberhead

at_ustp         aut  University of Applied Sciences St. Pölten (US | Cyber Security and Resilience
be_kuleuven     bel  KU Leuven                                     | Advanced Master of Cybersecurity
bg_nbu          not in CyberHEAD
hr_tvz          not in CyberHEAD
cy_ouc          cyp  Open University of Cyprus                     | Computer and Network Security
cz_ctu          not in CyberHEAD
dk_aau          dnk  Aalborg University                            | Cyber Security
ee_taltech      not in CyberHEAD
fi_jamk         fin  JAMK University of Applied Sciences           | Master's Degree in Information Technology, Cyber Securi
fr_ubs          fra  Université Bretagne Sud                       | Cyber-Sécurité des Systèmes Embarqués
de_hdbw         deu  HDBW - Hochschule der Bayerischen Wirtschaft  | Cyber Security
gr_ihu          grc  International Hellenic University             | MSc in Cybersecurity
hu_nke          hun  Ludovika University of Public Service         | Internati

{'entries_in_snapshot': 190,
 'programmes_in_cyberhead': 20,
 'not_in_cyberhead': ['bg_nbu',
  'cz_ctu',
  'ee_taltech',
  'hr_tvz',
  'lt_vilniustech',
  'lu_unilu',
  'nl_uva']}

In [6]:
selection = json.loads((SUPPLEMENTARY_FOLDER / "selection.json").read_text(encoding="utf-8"))
print(selection["rule"], "\n")


def select(excluded):
    taken, chosen = [], {}
    for role, candidates in selection["candidates"].items():
        eligible = [c for c in candidates if c["declares_role"] and c["code"] not in taken and c["code"] not in excluded]
        best = min(eligible, key=lambda c: len(c["declared"]))
        chosen[role] = best["code"]
        taken.append(best["code"])
    return chosen


by_rule = select(set())
after_exclusion = select({"LDR551"})
assert after_exclusion == {role: c["code"] for role, c in selection["chosen"].items()}

courses = {c["code"]: c for c in (json.loads(p.read_text(encoding="utf-8")) for p in sorted(COURSE_FOLDER.glob("*.json")))}
per_role = set(after_exclusion.values())
assert per_role <= set(courses)
extra = {code: course["declared_roles"] for code, course in courses.items() if code not in per_role}
for role in after_exclusion:
    print(f"{role:18s} {by_rule[role]:7s} -> {after_exclusion[role]}")
print()
for code, roles in sorted(extra.items()):
    print(f"{code:7s} {roles}  {courses[code].get('role_in_study')}")
sans = {"rule_first_for_ciso": by_rule["CISO"], "ciso_after_exclusion": after_exclusion["CISO"],
        "changed_by_exclusion": sorted(r for r in by_rule if by_rule[r] != after_exclusion[r]),
        "per_role_courses": len(per_role), "additional_courses": len(extra),
        "courses": len(courses), "in_database": trainings["courses"]}
sans

(provider's mapping only, no judgement by the authors):
  for each of the 12 ECSF roles, take the courses listed on SANS's own page for the role, in the order the page
  lists them (the featured course of the page banner, SEC536 "adversarial-ai", is not part of the list); keep those
  whose own course page declares the role among its ECSF "Relevant Job Roles"
  (`SansPageParser.declared_ecsf_roles`); take the one declaring the fewest ECSF roles (the most specific), the first
  in the page order on a tie, and never a course already taken for another role (roles are processed in the order
  of `ROLES`, which is the order of the ECSF role profiles document). 

CISO               LDR551  -> LDR521
IncidentResponder  FOR518  -> FOR518
LegalCompliance    SEC566  -> SEC566
ThreatIntel        SEC503  -> SEC503
Architect          SEC530  -> SEC530
Auditor            SEC275  -> SEC275
Educator           SEC403  -> SEC403
Implementer        SEC501  -> SEC501
Researcher         SEC497  -> SEC497
R

{'rule_first_for_ciso': 'LDR551',
 'ciso_after_exclusion': 'LDR521',
 'changed_by_exclusion': ['CISO'],
 'per_role_courses': 12,
 'additional_courses': 10,
 'courses': 22,
 'in_database': 22}

In [7]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output = OUTPUT_FOLDER / "02_data.json"
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "paper_section": "3.2",
    "run": RUN,
    "table_1": table_1,
    "table_s1": table_s1,
    "repeated_descriptions": repeated,
    "cyberhead": cyberhead,
    "sans_selection": sans,
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print("written to", output.relative_to(REPO_ROOT))

written to methodology\output\02_data.json
